# 13.2 The Call Stack and Frames

**Prerequisites:** 13.1 Objects, Names and the Heap, 4.1 Functions, 4.3 Generators  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What Python's call stack actually is — a chain of **frame objects**
- 🔴 **Proof that Python frames are not on the C stack** — measured, not asserted
- Walking the live stack with `sys._getframe()` and `f_back`
- Generator frames that outlive the call that made them
- 🔴 Why `RecursionError` fires **before** the limit you set, and why the gap moves
- `setrecursionlimit` — the classic footgun, and why 3.12+ defused it
- A traceback *is* the stack, printed outermost-first (**15.7**)

---

## Two stacks, and only one of them is yours

**13.1** established that every object lives on the heap. That raises an obvious question: if
even an `int` is heap-allocated, what is "the stack" in a Python traceback?

There are genuinely two stacks in a running Python process:

| Stack | Holds | Yours to reason about? |
|---|---|---|
| The **C stack** | CPython's own C function calls | no — an implementation detail |
| The **Python call stack** | one **frame object** per active Python call | 🔴 yes — this is what tracebacks show |

A **frame** is an ordinary heap object holding one call's local slots, its value stack, the
code object being run, and a pointer to the frame that called it. `RecursionError`, the
`O(depth)` memory cost of recursion (**14.12**), suspended generators (**4.3**) and every
traceback you have ever read are all consequences of this one structure.

The C/Java intuition says locals are stack-allocated and vanish when a function returns. In
CPython neither half is reliably true — and the rest of this notebook measures why.

## Walking the live stack

`sys._getframe()` hands you the frame you are currently executing in, and `f_back` walks
outward toward the caller. A request pipeline makes the chain easy to read.

In [ ]:
import sys


def current_stack():
    """Innermost frame first, walking outward via f_back.

    Stops at the module frame: what sits above it depends on how you ran
    this - a plain script, Jupyter, or a test runner each add their own.
    """
    chain, frame = [], sys._getframe()
    while frame is not None:
        chain.append(frame.f_code.co_name)
        if frame.f_code.co_name == "<module>":
            break
        frame = frame.f_back
    return chain


def handle_request(request_id):
    return authenticate(request_id)


def authenticate(request_id):
    return load_profile(request_id)


def load_profile(request_id):
    return current_stack()


print("the live call stack, innermost first:")
for depth, name in enumerate(handle_request("req-7")):
    print(f"  {depth}  {name}")

print("\neach frame is a real heap object:")
frame = sys._getframe()
print("  type            :", type(frame).__name__)
print("  its code object :", frame.f_code.co_name)
print("  has a caller?   :", frame.f_back is not None)

The chain reads exactly as the calls happened, from the inside out:
`current_stack` was called by `load_profile`, which was called by `authenticate`, then
`handle_request`, and finally `<module>` — the frame for the cell itself.

That last entry matters. **Module-level code runs in a frame too.** There is no special
"top level"; it is simply the outermost frame in the chain.

## 🔴 Proof: Python frames are not on the C stack

This is the claim worth measuring rather than believing. If frames lived on the thread's C
stack, then a thread with a **small** stack could not recurse as deeply as one with a large
stack. That is exactly how it works in C, and in Java.

So: raise Python's own recursion limit out of the way, then measure how deep recursion can
actually go in threads given C stacks from **256 KiB up to 8 MiB** — a 32× range.

In [ ]:
import threading

ORIGINAL_LIMIT = sys.getrecursionlimit()
sys.setrecursionlimit(100_000)          # take Python's own ceiling out of the way


def measure_depth():
    depth = 0

    def descend(n):
        nonlocal depth
        depth = n
        descend(n + 1)

    try:
        descend(0)
    except RecursionError:
        pass
    return depth


try:
    main_depth = measure_depth()
    print(f"  main thread (default stack)   : {main_depth:>8,} frames")

    results = {}
    for kib in (256, 512, 1024, 8192):
        threading.stack_size(kib * 1024)
        worker = threading.Thread(target=lambda k=kib: results.__setitem__(k, measure_depth()))
        worker.start()
        worker.join()
        print(f"  worker thread, {kib:>5} KiB stack : {results[kib]:>8,} frames")
    threading.stack_size(0)

    smallest = results[256]
    budget_bytes = 256 * 1024
    print(f"\n  If those {smallest:,} frames lived on the 256 KiB C stack, each would have")
    print(f"  to fit in {budget_bytes / smallest:.2f} bytes. A frame is far larger than that.")
    print("  -> Python frames are heap-allocated. The C stack is not the limit.")
finally:
    sys.setrecursionlimit(ORIGINAL_LIMIT)
    print(f"\n  recursion limit restored to {sys.getrecursionlimit()}")

Every thread reaches essentially the same depth — the 256 KiB thread gets
within a handful of frames of the 8 MiB one, despite a **32×** difference in C stack. If frames
were stack-allocated that result would be impossible.

The arithmetic printed above makes it undeniable: fitting that many frames into 256 KiB would
require each frame to occupy a couple of *bytes*. A CPython frame holds a code pointer, a
caller pointer, local slots and a value stack — orders of magnitude more.

🔴 **So Python's recursion limit is a policy, not a physical constraint.** CPython allocates
frames in chunks on a heap-backed *data stack* owned by the thread state, which is why the C
stack barely participates. The limit exists to turn runaway recursion into a catchable
exception instead of a crash.

> **Version note.** This is **3.11+** behaviour, from the "Faster CPython" work that made frames
> cheap and lazy. On 3.10 and earlier, frames were heavier and deep recursion in a small thread
> stack was genuinely dangerous.

## Frames outlive their calls — generators prove it

The C model says a function's locals die when it returns. A generator disproves that in the
most direct way available: suspend one mid-iteration and look at its frame.

In [ ]:
def stream_events(lines):
    """A log reader that keeps a running count between yields."""
    seen = 0
    for line in lines:
        seen += 1
        yield f"{seen:>3}: {line}"


stream = stream_events(["service=api status=200", "service=api status=500"])
print("before any iteration :", stream.gi_frame is not None)

print("first record         :", next(stream))
print("frame still alive    :", stream.gi_frame is not None)
print("its locals persist   :", {k: v for k, v in stream.gi_frame.f_locals.items()
                                 if k in ("seen", "line")})

print("second record        :", next(stream))
print("locals moved on      :", {k: v for k, v in stream.gi_frame.f_locals.items()
                                 if k in ("seen", "line")})

list(stream)                                    # drain it
print("\nafter exhaustion, gi_frame :", stream.gi_frame)
print("  ^ None - the frame is released once the generator finishes")

The generator's frame is created on the first `next()`, **survives between
calls** with `seen` and `line` intact, and is released only when the generator is exhausted —
at which point `gi_frame` becomes `None`.

This is the cleanest possible refutation of "locals live on the stack". These locals outlived
their call by an arbitrary amount of wall-clock time, and were resumed later. They live in a
heap object, because that is the only place they *could* live.

> The same mechanism powers `async` functions (**12.5**). An awaited coroutine is a suspended
> frame waiting to be resumed, which is why `await` is cheap and a thread is not.

## Why `RecursionError` fires early

Set the limit to 1000 and you do not get 1000 frames. You get slightly fewer, and the shortfall
is not a bug.

In [ ]:
def frames_available():
    depth = 0

    def descend(n):
        nonlocal depth
        depth = n
        descend(n + 1)

    try:
        descend(0)
    except RecursionError:
        pass
    return depth


def frames_already_on_the_stack():
    """Count every frame below us, whatever ran this cell."""
    count, frame = 0, sys._getframe()
    while frame is not None:
        count += 1
        frame = frame.f_back
    return count


limit = sys.getrecursionlimit()
reached = frames_available()
baseline = frames_already_on_the_stack()

print(f"  sys.getrecursionlimit()          : {limit}")
print(f"  frames the recursion managed     : {reached}")
print(f"  shortfall                        : {limit - reached}")
print(f"  frames already in use before it  : {baseline}")
print("\n  The limit counts ALL frames on the stack, not just recursive ones.")
print("  Run this same cell from a script, a test runner and Jupyter and the")
print("  shortfall changes, because each puts a different number of frames")
print("  underneath your code.")

The limit is a ceiling on **total stack depth**, not on how many times
your function may call itself. The cell, the helper and the recursive function are all already
on the stack when counting begins, so the recursion gets what is left over.

🔴 **This is why "increase the limit until it works" is a bad habit.** The number you need
depends on how deep the stack already was when the recursive call started, which changes with
call site, framework and test runner. Code that converts recursion to iteration (**14.12**) or
uses an explicit stack has no such dependency.

## `setrecursionlimit` — the footgun, and what changed

The classic warning is blunt: *raise the recursion limit too far and CPython will exhaust the
real C stack and segfault — no exception, no traceback, just a dead process.*

That warning was correct for years. It is worth checking whether it still holds, rather than
repeating it. The cell below sets the limit to **one million** in a **separate interpreter**, so
that whatever happens cannot take this notebook with it.

In [ ]:
import subprocess
import textwrap

EXPERIMENT = textwrap.dedent("""
    import sys
    sys.setrecursionlimit(1_000_000)      # far beyond any sane value

    def descend(n=0):
        return descend(n + 1)

    try:
        descend()
        print("completed without hitting any limit")
    except RecursionError as exc:
        print("RecursionError:", exc)
""")

done = subprocess.run([sys.executable, "-c", EXPERIMENT],
                      capture_output=True, text=True, encoding="utf-8",
                      errors="replace", timeout=300)

print("exit code :", done.returncode, "  (0 = clean exit, negative = killed by a signal)")
print("stdout    :", done.stdout.strip() or "(none)")
print("stderr    :", done.stderr.strip() or "(none)")
print()
if done.returncode == 0:
    print("  The interpreter survived and raised a catchable exception.")
else:
    print("  The interpreter died - this is the crash the old advice warns about.")

On this interpreter the child process **exits cleanly with a
`RecursionError`**, rather than crashing. Modern CPython keeps a separate guard on C-level
recursion depth, so `setrecursionlimit` can no longer be used to walk off the end of the C
stack quite so easily.

🔴 **This does not make raising the limit a good idea.** It converts a crash into an exception;
it does not make deep recursion correct or cheap. Each frame still costs memory (**14.12**), and
a million-deep recursion is a design problem, not a configuration problem.

> **Version note.** Treat the outcome above as *this interpreter's* behaviour, not a
> guarantee. Older CPython genuinely segfaults here, and C-level recursion — a recursive
> `__repr__`, a deeply nested structure being compared or pickled — goes through a different
> path than plain Python calls. If you are on 3.11 or earlier, assume the classic warning
> applies in full.

## A traceback *is* the stack

Everything above is what **15.7** is really reading when it teaches tracebacks. A traceback is
the frame chain, captured at the moment of the exception and printed outermost-first.

In [ ]:
import traceback


def parse_setting(raw):
    return int(raw.split("=")[1])          # raises on non-numeric input


def load_config(raw):
    return parse_setting(raw)


try:
    load_config("retries=three")
except ValueError as exc:
    print("exception:", type(exc).__name__, "-", exc)
    print("\nframes, outermost first (the order Python prints them):")
    for entry in traceback.extract_tb(exc.__traceback__):
        print(f"  {entry.name:16} line {entry.lineno}")
    print("\n  ^ same chain as sys._getframe()/f_back, in the opposite order:")
    print("    Python prints outermost-first so you read the story top to bottom,")
    print("    and the line that actually failed is LAST.")

The traceback lists `<module>`, then `load_config`, then `parse_setting`
— the exact reverse of the `f_back` walk from the first cell, because Python prints the story
from the beginning rather than from the crash.

🔴 **This is why you read a traceback from the bottom.** The last frame is where the exception
was raised; everything above it is how you got there. **15.7** builds on that, and **15.8**
puts `pdb` on the same frame chain — `pdb`'s `up` and `down` commands are literally walking
`f_back` and forward again.

---

## Common Mistakes & Pitfalls

1. 🔴 **Believing locals die when a function returns.** A generator or coroutine frame survives suspension with its locals intact.
2. 🔴 **Assuming the recursion limit equals available depth.** Frames already on the stack count toward it; you always get fewer.
3. **Raising `setrecursionlimit` until the error stops.** It hides a design problem and the number you need is not stable across call sites.
4. **Thinking a bigger thread stack allows deeper Python recursion.** Measured above: it does not, because frames are on the heap.
5. **Reading a traceback from the top.** The failure is at the *bottom* (**15.7**).
6. **Using `sys._getframe()` in production code.** It is fine for debugging and logging helpers, and a maintenance hazard anywhere else.
7. **Editing `frame.f_locals` and expecting it to stick** in a function frame — see **13.1** and PEP 667.
8. **Forgetting recursion costs O(depth) memory.** Every frame is a live heap object until it returns (**14.12**).
9. **Assuming C-level recursion obeys the same limit.** A recursive `__repr__` or comparison takes a different path.

## Best Practices

- Convert deep recursion to iteration with an explicit stack (**14.12**) rather than raising the limit.
- Treat the recursion limit as a runaway-detector, not a tuning knob.
- Use `traceback.extract_tb` when you need the frame chain as data — for structured logging, for example (**15.10**).
- Remember a suspended generator holds its frame, and therefore everything its locals reference (**13.3**).
- Prefer `frame.f_locals` over `locals()` for a live view while debugging (**13.1**).
- Let `pdb`'s `up`/`down` walk the chain for you instead of printing frames by hand (**15.8**).
- Keep recursion depth proportional to `log n` where you can — balanced trees recurse safely, linked lists do not (**14.4**, **14.7**).

## Practice Exercises

Try these before moving on.

1. Print the frame chain from inside a function called three levels deep. Now call it from a comprehension — does an extra frame appear? (Check the 3.12 version note.)
2. Measure the real available depth in your test runner. Is it the same as at a bare prompt? Explain the difference.
3. 🔴 Write a recursive directory walk that dies with `RecursionError` on a deep tree, then rewrite it with an explicit stack. Compare peak memory with `tracemalloc` (**17.5**).
4. Suspend a generator holding a large list, then check with `getrefcount` (**13.1**) that the list is still alive. Explain why this can leak.
5. Repeat the thread-stack experiment with a 64 KiB stack. Does the depth change? What does that tell you?
6. Catch an exception, store `exc.__traceback__`, and re-raise it later. What is still referenced while you hold it? (**13.3** — this is a real leak source.)
7. Take a traceback from a service log and identify, without running anything, which frame raised and which merely propagated.
8. **Interview question:** *“Why does Python have a recursion limit when C does not?”* Answer using frames, the heap, and what the limit is actually protecting.

---

## Version notes

| Version | Change |
|---|---|
| **3.11** | 🔴 **Cheaper, lazier frames** — frames are allocated in chunks on a heap-backed data stack, and frame *objects* are only materialised when something asks for them |
| **3.11** | Fine-grained error locations in tracebacks — the `^^^^` markers under the failing expression (**15.7**) |
| **3.12** | Comprehensions are inlined (PEP 709), so a list comprehension no longer adds a frame |
| **3.12** | A separate C-recursion guard, which is why `setrecursionlimit` is far harder to crash with than it used to be |
| **3.13** | PEP 667 — `frame.f_locals` is the live view; `locals()` is the snapshot (**13.1**) |

> 🔴 **CPython specifics again.** Frame layout, the data stack and the recursion limit are
> implementation details. PyPy inlines aggressively and may not produce a frame per call at all.
> Depend on the *semantics* — tracebacks, generators resuming correctly — never on the numbers.

## 13 How Python Works Under the Hood — the folder

| Notebook | Covers |
|---|---|
| **13.1** | objects on the heap, names, identity, interning, immortality |
| **13.2** | this notebook — the call stack, frames, recursion limits, tracebacks |
| **13.3** | reference counting, cycles, the garbage collector, `weakref` |
| **13.4** | where the memory actually goes — `getsizeof`, container overhead, `__slots__` measured |
| **13.5** | attribute lookup and class creation — descriptors, `__init_subclass__`, metaclasses |

**The one-sentence version:** *the call stack is a chain of heap-allocated frame objects, which
is why generators can pause, why recursion has a limit, and why a traceback can be printed at
all.*

## Related

- **4.3** Function Generators — the frames that pause
- **12.5** asyncio — the same suspension mechanism, scheduled
- **14.12** Recursion and Backtracking — the O(depth) cost this explains
- **15.7** Reading Failures — tracebacks in depth
- **15.8** The Interactive Debugger — `up`/`down` walk this chain